# Silver PSA Economy

This notebook turns the PSA provincial accounts Bronze tables into economic measures per **reporting unit and year** (2019–2023).

| Measure | Source | Bronze table | Unit (from the table title) |
| --- | --- | --- | --- |
| `gdp_total_current` | DS_27, Table 3.72 | `psa_ds_27` | thousand PhP, current prices |
| `gdp_total_constant` | DS_28, Table 3.73 | `psa_ds_28` | thousand PhP, constant 2018 prices |
| `gdp_per_capita_current` | DS_29, Table 3.74 | `psa_ds_29` | PhP, current prices |
| `afs_gva_current` | DS_31, Table 3.94 | `psa_ds_31` | thousand PhP, current prices |
| `afs_gva_constant` | DS_32, Table 3.95 | `psa_ds_32` | thousand PhP, constant 2018 prices |
| (reconciliation only) | DS_26, Table 3.63 (A&F GVA by region, 2022–2024) | `psa_ds_26` | thousand PhP |

| Silver table | One row = |
| --- | --- |
| `map_psa_area_to_psgc` | one PSA row label (134 labels in DS_27–DS_31; DS_32 has 133, with no City of Tacloban row), loaded from the reviewed seed |
| `psa_economy_row` | one PSA value: dataset × printed row × year, with its role and unit (traceability, reconciliation) |
| `psa_economy_unit_year` | one reporting unit per year (100 × 5) with the five measures |
| `ref_indicator` | one measure, with its source table, unit and price basis (D17) |

**Rules applied** (decision log and schema doc #43):

* Rows are classified by their **cleaned name** matched to the seed, never by the number of leading dots, which differs between tables (D13).
* PSA province rows **exclude their HUCs**; HUCs are separate rows and separate units (DL-010). NCR uses the NCR row; its city rows are validation only (DL-012).
* **Lucena** is added to Quezon for the additive measures (DL-011). Quezon's per-capita GDP is recomputed as `(GDP Quezon + GDP Lucena) ÷ (population Quezon + population Lucena)`, using the population PSA itself used for each published per-capita value (`GDP x 1,000 ÷ per capita`). Those populations equal the DS_18 projections (e.g. Lucena 2020 = 286,191), so this matches D16.
* Values are kept **as published**. A value that is not published stays NULL, except when it can be derived exactly: if a single row is missing from a printed region, its value is the region total minus the other rows (e.g. City of Tacloban in DS_32). Such values are listed in `derived_values` and in check DQ-PSA-08.
* Units and price basis come from the table titles (D17): Bronze skips the title row, so they are recorded once in `ref_indicator`.

Every data-quality check runs **before** writing. If a `BLOCK` check fails, results go to `dq_results`, no Silver table is written, and the notebook raises an error.

## Configuration

In [0]:
from pyspark.sql import functions as F
import uuid, os, re, hashlib
from datetime import datetime
import pandas as pd

# ── Unity Catalog targets ────────────────────────────────────────
CATALOG = "tuklas_dev"
BRONZE_SCHEMA = "02_bronze"
SILVER_SCHEMA = "03_silver"
DQ_SCHEMA = "06_data_quality"
NOTEBOOK_NAME = "03_silver_psa_economy"

run_id = f"silver_psa_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

# ── Inputs ───────────────────────────────────────────────────────
# measure -> (dataset_id, bronze table, PSA table, title as published, unit, price basis, additive?)
MEASURES = {
    "gdp_total_current":      ("DS_27", "psa_ds_27", "3.72", "Gross Domestic Product, by Province: 2019 to 2023 (In thousand PhP: at current prices)", "thousand PhP", "current prices", True),
    "gdp_total_constant":     ("DS_28", "psa_ds_28", "3.73", "Gross Domestic Product, by Province: 2019 to 2023 (In thousand PhP: at constant prices)", "thousand PhP", "constant 2018 prices", True),
    "gdp_per_capita_current": ("DS_29", "psa_ds_29", "3.74", "Per Capita Gross Domestic Product, by Province and HUC: 2019 to 2023 (In PhP: at current prices)", "PhP", "current prices", False),
    "afs_gva_current":        ("DS_31", "psa_ds_31", "3.94", "Gross Value Added in Accommodation and Food Service Activities, by Province and HUC: 2019 to 2023 (In thousand PhP: at current prices)", "thousand PhP", "current prices", True),
    "afs_gva_constant":       ("DS_32", "psa_ds_32", "3.95", "Gross Value Added in Accommodation and Food Service Activities, by Province and HUC: 2019 to 2023 (In thousand PhP: at constant prices)", "thousand PhP", "constant 2018 prices", True),
}
RECON_TABLE = "psa_ds_26"     # Table 3.63, A&F GVA by region 2022-2024 (reconciliation only)
YEARS = [2019, 2020, 2021, 2022, 2023]
TBL_UNIT = "geo_reporting_unit"   # 01_silver_geography

SEED_RELATIVE_PATH = "resources/seeds/map_psa_area_to_psgc.csv"
SEED_PATH_OVERRIDE = None

QUEZON, LUCENA_LABEL = "0405600000", "City of Lucena"
ROUNDING_TOLERANCE = 5          # printed totals vs sum of rows may differ by rounding (same unit as the table)
DS26_TOLERANCE_PCT = 2.0        # DQ-PSA-06: Table 3.63 vs DS_31/DS_32 national sum (different release; revisions expected)

# ── Output tables ────────────────────────────────────────────────
TBL_MAP = "map_psa_area_to_psgc"
TBL_ROWS = "psa_economy_row"
TBL_UNIT_YEAR = "psa_economy_unit_year"
TBL_REF = "ref_indicator"

def bronze_table(name): return f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`{name}`"
def silver_table(name): return f"`{CATALOG}`.`{SILVER_SCHEMA}`.`{name}`"
def dq_table(name): return f"`{CATALOG}`.`{DQ_SCHEMA}`.`{name}`"

print(f"Run ID: {run_id}")

## Helpers

Same pattern as the other Silver notebooks: `dq(...)` records each check (written to `dq_results` at the end), and `write_silver(...)` overwrites a whole table.

In [0]:
dq_results = []

def dq(check_id, table_name, severity, failed_rows, expected, observed, sample_keys=None, notes=""):
    """Record one data-quality check result. failed_rows = 0 means PASS."""
    status = "PASS" if failed_rows == 0 else "FAIL"
    keys = " | ".join(str(k) for k in list(sample_keys)[:20]) if sample_keys else None
    dq_results.append((run_id, NOTEBOOK_NAME, check_id, table_name, severity, status, int(failed_rows),
                       str(expected), str(observed), keys, notes))
    marker = "✓" if status == "PASS" else ("✗" if severity == "BLOCK" else "!")
    print(f"{marker} {check_id:<11} {severity:<6} {status:<4} | {table_name} | expected {expected} | observed {observed}"
          + (f" | e.g. {keys}" if keys else ""))

def sample(df, *cols, n=20):
    return [" / ".join("" if r[c] is None else str(r[c]) for c in cols) for r in df.select(*cols).limit(n).collect()]

def write_silver(df, name):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(silver_table(name)))
    return spark.table(silver_table(name)).count()

def find_seed_path():
    if SEED_PATH_OVERRIDE:
        return SEED_PATH_OVERRIDE
    here = os.getcwd()
    for _ in range(8):
        candidate = os.path.join(here, SEED_RELATIVE_PATH)
        if os.path.exists(candidate):
            return candidate
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    raise FileNotFoundError(f"{SEED_RELATIVE_PATH} not found above {os.getcwd()}. Run from the repo Git folder or set SEED_PATH_OVERRIDE.")

def clean_label(c):
    """PSA labels carry 0-4 leading dots depending on the table; the name itself is the key (D13)."""
    return F.trim(F.regexp_replace(F.trim(c), r"^\.+", ""))

spark.sql(f"""
INSERT INTO {silver_table('silver_run_log')}
(run_id, notebook_name, started_at, completed_at, tables_written, rows_written,
 dq_checks_run, dq_block_failures, dq_review_flags, status, notes)
VALUES (:run_id, :nb, current_timestamp(), NULL, 0, 0, 0, 0, 0, 'STARTED', 'Notebook execution started')
""", args={"run_id": run_id, "nb": NOTEBOOK_NAME})
print(f"✓ STARTED record inserted for {run_id}")

## Step 1 — `map_psa_area_to_psgc`: load the PSA crosswalk seed

The seed `resources/seeds/map_psa_area_to_psgc.csv` has one row per PSA label (dots removed), reviewed like code.

| Seed column | Meaning |
| --- | --- |
| `psa_label` | Row name as printed, without leading dots (two NCR cities print the "ñ" as a replacement character; they are matched as printed) |
| `psa_region_code` | Region the row is printed under (PSA's 2019–2023 tables use 17 regions: Negros provinces under VI/VII, Sulu under BARMM) |
| `row_role` | `unit` (the unit's value), `add_to_unit` (summed into another unit: Lucena → Quezon), `ncr_component` (NCR city, inside the NCR row), `region_total` (validation) |
| `unit_psgc_code` | Unit the value belongs to |
| `area_psgc_code`, `area_psgc_name` | PSGC code and name of the printed area |

In [0]:
seed_path = find_seed_path()
seed_bytes = open(seed_path, "rb").read()
seed_sha256 = hashlib.sha256(seed_bytes).hexdigest()
seed_pd = pd.read_csv(seed_path, dtype=str, keep_default_na=False, encoding="utf-8")
seed_pd = seed_pd.apply(lambda s: s.str.strip()).replace({"": None})
psa_map = (spark.createDataFrame(seed_pd.astype(object).where(seed_pd.notna(), None),
                                 schema=", ".join(f"`{c}` STRING" for c in seed_pd.columns))
           .withColumn("_seed_file", F.lit(SEED_RELATIVE_PATH))
           .withColumn("_seed_sha256", F.lit(seed_sha256))
           .withColumn("_silver_run_id", F.lit(run_id))
           .withColumn("_processed_at", F.current_timestamp()))
print(f"Seed: {seed_path} | rows: {len(seed_pd)} | sha256: {seed_sha256[:12]}…")
display(psa_map.groupBy("row_role").count().orderBy("row_role"))

# DQ-PSA-10, before any join: one seed row per label, so the label join can never duplicate Bronze rows
dup_labels = psa_map.groupBy("psa_label").count().filter("count > 1")
dq("DQ-PSA-10", TBL_MAP, "BLOCK", dup_labels.count(), "each psa_label once in the seed",
   f"{dup_labels.count()} labels repeated", sample(dup_labels, "psa_label", "count"),
   notes="A repeated label would duplicate Bronze rows in the join and inflate unit totals.")

## Step 2 — `psa_economy_row`: one value per dataset, row and year

Each Bronze table has a label column (named `Provinces` or `Region` depending on the table) and one text column per year. Labels are cleaned (leading dots removed) and joined to the seed; values are cast to numbers (empty stays NULL).

In [0]:
def load_measure(measure):
    ds, tbl, *_ = MEASURES[measure]
    t = spark.table(bronze_table(tbl))
    data_cols = [c for c in t.columns if not c.startswith("_") and c != "source_row_number"]
    label_col = data_cols[0]
    year_cols = [c for c in data_cols if re.fullmatch(r"\d{4}", c)]
    assert sorted(int(y) for y in year_cols) == YEARS, f"{tbl}: expected year columns {YEARS}, found {year_cols}"
    stack = ", ".join(f"'{y}', `{y}`" for y in year_cols)
    return (t.select(clean_label(F.col(f"`{label_col}`")).alias("psa_label"),
                     F.col(f"`{label_col}`").alias("label_as_printed"),
                     F.expr(f"stack({len(year_cols)}, {stack}) AS (year, value_text)"),
                     "_source_dataset_id", "_source_file", "_source_sha256", F.col("_run_id").alias("_bronze_run_id"),
                     *(["source_row_number"] if "source_row_number" in t.columns else []))
             .withColumn("measure", F.lit(measure))
             .withColumn("dataset_id", F.lit(ds))
             .withColumn("year", F.col("year").cast("int"))
             .withColumn("value_text", F.trim("value_text"))
             .withColumn("value", F.expr("try_cast(regexp_replace(value_text, ',', '') AS DOUBLE)")))

raw = None
for m in MEASURES:
    df = load_measure(m)
    if "source_row_number" not in df.columns:
        df = df.withColumn("source_row_number", F.lit(None).cast("string"))
    raw = df if raw is None else raw.unionByName(df)

m_ = psa_map.select("psa_label", "psa_region_code", "row_role", "unit_psgc_code", "area_psgc_code", "area_psgc_name")
rows = (raw.join(m_, "psa_label", "left")
        .withColumn("_silver_run_id", F.lit(run_id))
        .withColumn("_processed_at", F.current_timestamp()))
display(rows.groupBy("dataset_id", "row_role").count().orderBy("dataset_id", "row_role"))

## Step 3 — Fill a single missing row from its printed region total

For each measure, year and printed region: if exactly one `unit` / `add_to_unit` row of that region is **absent** from the table, and the region total is printed, the absent value is `region total − sum of the other rows`. Province and HUC rows add up to the region totals (checked in DQ-PSA-03), so this reproduces the value PSA used in its own total. Everything else that is missing stays NULL.

A derived row has no printed row of its own, so `source_row_number` stays NULL. Its lineage (`_source_file`, `_source_sha256`, `_bronze_run_id`) comes from the printed region-total row it was calculated from; that row's number is kept in `derivation_source_row_number`, and `derivation_method` says how the value was obtained.

In [0]:
expected_children = psa_map.filter(F.col("row_role").isin("unit", "add_to_unit")).select("psa_label", "psa_region_code", "row_role", "unit_psgc_code", "area_psgc_code", "area_psgc_name")
grid = expected_children.crossJoin(spark.createDataFrame([(m, MEASURES[m][0], y) for m in MEASURES for y in YEARS],
                                                         "measure STRING, dataset_id STRING, year INT"))
present = rows.filter(F.col("row_role").isin("unit", "add_to_unit")).select("psa_label", "measure", "year")
absent = grid.join(present, ["psa_label", "measure", "year"], "left_anti")
n_absent = absent.groupBy("measure", "year", "psa_region_code").count()

region_tot = (rows.filter(F.col("row_role") == "region_total")
              .select("measure", "year", "psa_region_code", F.col("value").alias("region_value"),
                      # lineage of a derived value = the printed region-total row it comes from
                      "_source_file", "_source_sha256", "_bronze_run_id",
                      F.col("source_row_number").alias("derivation_source_row_number")))
children_sum = (rows.filter(F.col("row_role").isin("unit", "add_to_unit"))
                .groupBy("measure", "year", "psa_region_code")
                .agg(F.sum("value").alias("children_value"),
                     # F.sum skips NULLs, so only derive when every other row in the region has a value
                     F.sum(F.when(F.col("value").isNull(), 1).otherwise(0)).alias("n_null_children")))
derived = (absent.join(n_absent.filter("count = 1").drop("count"), ["measure", "year", "psa_region_code"])
           .join(region_tot, ["measure", "year", "psa_region_code"])
           .join(children_sum.filter("n_null_children = 0").drop("n_null_children"), ["measure", "year", "psa_region_code"])
           .withColumn("value", F.col("region_value") - F.col("children_value"))
           .withColumn("label_as_printed", F.lit(None).cast("string"))
           .withColumn("value_text", F.lit(None).cast("string"))
           .withColumn("_source_dataset_id", F.col("dataset_id"))
           .withColumn("source_row_number", F.lit(None).cast("string"))     # never printed
           .withColumn("derivation_method", F.lit("region total minus the other printed rows of the region (DL-024)"))
           .withColumn("is_derived", F.lit(True))
           .withColumn("_silver_run_id", F.lit(run_id))
           .withColumn("_processed_at", F.current_timestamp()))
# per-capita is not additive, so it is never derived from a region total
additive = [m for m, v in MEASURES.items() if v[6]]
derived = derived.filter(F.col("measure").isin(additive)).drop("region_value", "children_value")

DERIVATION_COLS = ["derivation_method", "derivation_source_row_number"]
rows_all = (rows.withColumn("is_derived", F.lit(False))
                .withColumn("derivation_method", F.lit(None).cast("string"))
                .withColumn("derivation_source_row_number", F.lit(None).cast("string"))
                .unionByName(derived.select(*rows.columns, "is_derived", *DERIVATION_COLS)))
display(derived.select("measure", "year", "psa_label", "value", "derivation_source_row_number", "_source_file").orderBy("measure", "year"))

## Step 4 — `psa_economy_unit_year`: one row per unit and year

* Additive measures: `unit` rows plus `add_to_unit` rows (Lucena into Quezon).
* `gdp_per_capita_current`: the `unit` row as published, except Quezon, which is recomputed with Lucena (see the top of the notebook).
* `derived_values` lists any value filled from a region total.

In [0]:
loaded = rows_all.filter(F.col("row_role").isin("unit", "add_to_unit"))

wide = None
for m, (ds, tbl, ptab, title, unit, basis, is_add) in MEASURES.items():
    part = loaded.filter(F.col("measure") == m)
    if not is_add:
        part = part.filter(F.col("row_role") == "unit")
    agg = (part.groupBy("unit_psgc_code", "year")
           .agg(F.sum("value").alias(m),
                F.concat_ws("; ", F.collect_set(F.when(F.col("is_derived"), F.concat(F.lit(f"{m} ({ds}) "), F.col("psa_label"), F.lit(" from region total"))))).alias(f"_d_{m}")))
    wide = agg if wide is None else wide.join(agg, ["unit_psgc_code", "year"], "full")

# Quezon per-capita: (GDP Q + GDP L) * 1000 / (pop Q + pop L), pop = GDP * 1000 / published per capita
pc = (rows_all.filter((F.col("measure").isin("gdp_total_current", "gdp_per_capita_current"))
                      & ((F.col("unit_psgc_code") == QUEZON)))
      .groupBy("psa_label", "year").pivot("measure", ["gdp_total_current", "gdp_per_capita_current"]).agg(F.first("value"))
      .withColumn("implied_pop", F.col("gdp_total_current") * 1000 / F.col("gdp_per_capita_current"))
      .groupBy("year").agg(F.sum("gdp_total_current").alias("_g"), F.sum("implied_pop").alias("_p"), F.count("*").alias("_n"))
      .withColumn("_pc_quezon", F.round(F.col("_g") * 1000 / F.col("_p"))))
wide = (wide.join(pc.select("year", "_pc_quezon", "_n").withColumn("unit_psgc_code", F.lit(QUEZON)), ["unit_psgc_code", "year"], "left")
        .withColumn("gdp_per_capita_current", F.when((F.col("unit_psgc_code") == QUEZON) & (F.col("_n") == 2), F.col("_pc_quezon"))
                                                .otherwise(F.col("gdp_per_capita_current"))))

units = spark.table(silver_table(TBL_UNIT)).select("unit_psgc_code", "unit_name", "unit_type", "region_code", "region_name")
ugrid = units.crossJoin(spark.createDataFrame([(y,) for y in YEARS], "year INT"))
lineage = (rows_all.filter(F.col("row_role").isin("unit", "add_to_unit") & ~F.col("is_derived"))
           .groupBy("unit_psgc_code", "year")
           .agg(F.concat_ws("; ", F.array_sort(F.collect_set("_source_dataset_id"))).alias("_source_dataset_id"),
                F.concat_ws("; ", F.array_sort(F.collect_set("_source_sha256"))).alias("_source_sha256"),
                F.concat_ws("; ", F.array_sort(F.collect_set("_bronze_run_id"))).alias("_bronze_run_id")))
derived_cols = [F.col(f"_d_{m}") for m in MEASURES]
unit_year = (ugrid.join(wide, ["unit_psgc_code", "year"], "left").join(lineage, ["unit_psgc_code", "year"], "left")
    .withColumn("derived_values", F.concat_ws("; ", *[F.when(c != "", c) for c in derived_cols]))
    .withColumn("derived_values", F.when(F.col("derived_values") == "", None).otherwise(F.col("derived_values")))
    .withColumn("per_capita_recomputed", F.col("unit_psgc_code") == QUEZON)
    .withColumn("_silver_run_id", F.lit(run_id))
    .withColumn("_processed_at", F.current_timestamp())
    .select("unit_psgc_code", "unit_name", "unit_type", "region_code", "region_name", "year",
            *[F.col(m).cast("decimal(20,0)").alias(m) for m in MEASURES],   # whole numbers, as published
            "derived_values", "per_capita_recomputed", "_source_dataset_id", "_source_sha256", "_bronze_run_id",
            "_silver_run_id", "_processed_at"))

ref_indicator = spark.createDataFrame(
    [(m, v[0], v[1], v[2], v[3], v[4], v[5], "additive" if v[6] else "ratio (non-additive)") for m, v in MEASURES.items()],
    "measure STRING, dataset_id STRING, bronze_table STRING, psa_table STRING, title STRING, unit STRING, price_basis STRING, additivity STRING"
    ).withColumn("_silver_run_id", F.lit(run_id)).withColumn("_processed_at", F.current_timestamp())
display(unit_year.filter(F.col("unit_psgc_code").isin(QUEZON, "0831600000", "1300000000")).orderBy("unit_name", "year"))

## Step 5 — Data-quality checks

| ID | Check | Severity |
| --- | --- | --- |
| DQ-PSA-01 | Every Bronze label is in the seed | BLOCK |
| DQ-PSA-02 | Seed: every reporting unit has exactly one `unit` row; every `unit` / `add_to_unit` code is a reporting unit | BLOCK |
| DQ-COM-06 | Every published value is a number | BLOCK |
| DQ-COM-03 | One row per unit and year (100 × 5) | BLOCK |
| DQ-PSA-03 | For every dataset, year and printed region: region total = sum of its rows (± rounding) | BLOCK |
| DQ-PSA-04 | NCR row = sum of NCR city rows (± rounding) | BLOCK |
| DQ-PSA-05 | Every unit-year has all five measures (after derivation) | REVIEW |
| DQ-PSA-06 | DS_26 (Table 3.63) national A&F GVA 2022–2023 vs the DS_31 / DS_32 sum | REVIEW |
| DQ-PSA-07 | Quezon's recomputed per-capita lies between the published Quezon and Lucena values | BLOCK |
| DQ-PSA-08 | Values derived from a region total are listed | LOG |
| DQ-PSA-09 | A&F GVA ≤ GDP for every unit-year (current and constant) | BLOCK |
| DQ-COM-09 | No replacement characters (`�`) in unit names | BLOCK |
| DQ-PSA-10 | Each `psa_label` appears once in the seed (checked in Step 1, before the join) | BLOCK |

In [0]:
unmapped = rows.filter(F.col("row_role").isNull()).select("dataset_id", "label_as_printed").distinct()
dq("DQ-PSA-01", TBL_ROWS, "BLOCK", unmapped.count(), "every label in the seed", f"{unmapped.count()} unmapped labels",
   sample(unmapped, "dataset_id", "label_as_printed"),
   notes="Add new or renamed labels to resources/seeds/map_psa_area_to_psgc.csv.")

unit_rows = psa_map.filter("row_role = 'unit'").groupBy("unit_psgc_code").count()
unit_codes = [r[0] for r in units.select("unit_psgc_code").collect()]
dup_units = unit_rows.filter("count != 1")
missing_units = units.join(unit_rows, "unit_psgc_code", "left_anti")
bad_codes = psa_map.filter(F.col("row_role").isin("unit", "add_to_unit") & ~F.col("unit_psgc_code").isin(unit_codes))
n_bad = dup_units.count() + missing_units.count() + bad_codes.count()
dq("DQ-PSA-02", TBL_MAP, "BLOCK", n_bad, "each of the units has exactly 1 unit row",
   f"{dup_units.count()} duplicated, {missing_units.count()} missing, {bad_codes.count()} unknown codes",
   sample(dup_units, "unit_psgc_code") + sample(missing_units, "unit_name") + sample(bad_codes, "psa_label"))

bad_val = rows.filter(F.col("value_text").isNotNull() & (F.col("value_text") != "") & F.col("value").isNull())
dq("DQ-COM-06", TBL_ROWS, "BLOCK", bad_val.count(), "every published value is a number", f"{bad_val.count()} non-numeric",
   sample(bad_val, "dataset_id", "psa_label", "year", "value_text"))

dup = unit_year.groupBy("unit_psgc_code", "year").count().filter("count > 1")
n_uy = unit_year.count()
dq("DQ-COM-03", TBL_UNIT_YEAR, "BLOCK", dup.count() + (0 if n_uy == len(unit_codes) * len(YEARS) else 1),
   f"{len(unit_codes)} units x {len(YEARS)} years", f"{n_uy} rows, {dup.count()} duplicates")

children = (rows_all.filter(F.col("row_role").isin("unit", "add_to_unit") & F.col("measure").isin(additive))
            .groupBy("measure", "year", "psa_region_code").agg(F.sum("value").alias("children")))
region_chk = (rows.filter(F.col("row_role") == "region_total").select("measure", "year", "psa_region_code", F.col("value").alias("printed"))
              .join(children, ["measure", "year", "psa_region_code"], "left")
              .withColumn("diff", F.col("printed") - F.col("children")))
region_bad = region_chk.filter(F.abs("diff") > ROUNDING_TOLERANCE)
dq("DQ-PSA-03", TBL_ROWS, "BLOCK", region_bad.count(), f"|region total - rows| <= {ROUNDING_TOLERANCE}",
   f"{region_chk.count()} region-years checked, {region_bad.count()} outside tolerance; max |diff| "
   f"{region_chk.agg(F.max(F.abs('diff'))).collect()[0][0]}",
   sample(region_bad, "measure", "year", "psa_region_code", "diff"))

ncr_parts = (rows.filter((F.col("row_role") == "ncr_component") & F.col("measure").isin(additive))
             .groupBy("measure", "year").agg(F.sum("value").alias("parts")))
ncr_chk = (rows.filter((F.col("row_role") == "unit") & (F.col("unit_psgc_code") == "1300000000") & F.col("measure").isin(additive))
           .select("measure", "year", F.col("value").alias("ncr")).join(ncr_parts, ["measure", "year"])
           .withColumn("diff", F.col("ncr") - F.col("parts")))
ncr_bad = ncr_chk.filter(F.abs("diff") > ROUNDING_TOLERANCE)
dq("DQ-PSA-04", TBL_ROWS, "BLOCK", ncr_bad.count(), f"|NCR - cities| <= {ROUNDING_TOLERANCE}", f"{ncr_bad.count()} outside tolerance",
   sample(ncr_bad, "measure", "year", "diff"))

null_any = " OR ".join(f"{m} IS NULL" for m in MEASURES)
incomplete = unit_year.filter(F.expr(null_any))
dq("DQ-PSA-05", TBL_UNIT_YEAR, "REVIEW", incomplete.count(), "all five measures for every unit-year", f"{incomplete.count()} incomplete",
   sample(incomplete, "unit_name", "year"))

ds26_note = "psa_ds_26 not found"
ds26_bad = []
try:
    t26 = spark.table(bronze_table(RECON_TABLE))
    lab26 = [c for c in t26.columns if not c.startswith("_") and c != "source_row_number"][0]
    nat = t26.filter(clean_label(F.col(f"`{lab26}`")) == "Philippines").collect()[0].asDict()
    def _col26(basis, y):
        return next(v for k, v in nat.items() if re.search(basis, k, re.I) and k.endswith(str(y)))
    ds26_note = []
    for y in (2022, 2023):
        for m, basis in (("afs_gva_current", "current"), ("afs_gva_constant", "constant")):
            ours = float(unit_year.filter(F.col("year") == y).agg(F.sum(m)).collect()[0][0])
            theirs = float(str(_col26(basis, y)).replace(",", ""))
            pct = (ours - theirs) / theirs * 100
            ds26_note.append(f"{m} {y}: {pct:+.2f}%")
            if abs(pct) > DS26_TOLERANCE_PCT:
                ds26_bad.append(f"{m} {y}: {pct:+.2f}%")
    ds26_note = "; ".join(ds26_note)
except Exception as e:
    ds26_bad, ds26_note = ["not checked"], f"not checked: {e}"
dq("DQ-PSA-06", RECON_TABLE, "REVIEW", len(ds26_bad), f"national A&F within {DS26_TOLERANCE_PCT}% of Table 3.63", ds26_note, ds26_bad,
   notes="Table 3.63 is a later release (2022-2024); small revisions are expected.")

pubq = (rows.filter((F.col("measure") == "gdp_per_capita_current") & (F.col("unit_psgc_code") == QUEZON))
        .groupBy("year").agg(F.min("value").alias("lo"), F.max("value").alias("hi")))
q_bad = (unit_year.filter(F.col("unit_psgc_code") == QUEZON).select("year", "gdp_per_capita_current").join(pubq, "year")
         .filter(~F.col("gdp_per_capita_current").between(F.col("lo"), F.col("hi"))))
dq("DQ-PSA-07", TBL_UNIT_YEAR, "BLOCK", q_bad.count(), "Quezon per capita between published Quezon and Lucena values",
   f"{q_bad.count()} years outside", sample(q_bad, "year", "gdp_per_capita_current", "lo", "hi"))

der = rows_all.filter("is_derived")
dq("DQ-PSA-08", TBL_ROWS, "LOG", 0, "derived values listed", f"{der.count()} values derived from a region total",
   sample(der, "dataset_id", "psa_label", "year", "value"))

over = unit_year.filter((F.col("afs_gva_current") > F.col("gdp_total_current")) | (F.col("afs_gva_constant") > F.col("gdp_total_constant")))
dq("DQ-PSA-09", TBL_UNIT_YEAR, "BLOCK", over.count(), "A&F GVA <= GDP", f"{over.count()} violations", sample(over, "unit_name", "year"))

bad_chars = unit_year.filter(F.col("unit_name").contains("�"))
dq("DQ-COM-09", TBL_UNIT_YEAR, "BLOCK", bad_chars.count(), "0 unit names with �", f"{bad_chars.count()}")

dq_df = spark.createDataFrame(dq_results,
    "run_id STRING, notebook_name STRING, check_id STRING, table_name STRING, severity STRING, status STRING, "
    "failed_rows BIGINT, expected STRING, observed STRING, sample_keys STRING, notes STRING"
    ).withColumn("checked_at", F.current_timestamp())
display(dq_df.select("check_id", "severity", "status", "failed_rows", "expected", "observed", "sample_keys"))

## Step 6 — Write Silver tables and update the run log

In [0]:
dq_df.select("run_id", "notebook_name", "check_id", "table_name", "severity", "status", "failed_rows",
             "expected", "observed", "sample_keys", "checked_at", "notes") \
     .write.format("delta").mode("append").saveAsTable(dq_table("dq_results"))

block_failures = [r for r in dq_results if r[4] == "BLOCK" and r[5] == "FAIL"]
review_flags = [r for r in dq_results if r[4] == "REVIEW" and r[5] == "FAIL"]

def update_run_log(status, tables_written, rows_written, notes):
    spark.sql(f"""
    UPDATE {silver_table('silver_run_log')}
    SET status = :status, completed_at = current_timestamp(), tables_written = :tw, rows_written = :rw,
        dq_checks_run = :dc, dq_block_failures = :db, dq_review_flags = :dr, notes = :notes
    WHERE run_id = :run_id
    """, args={"status": status, "tw": tables_written, "rw": rows_written, "dc": len(dq_results),
               "db": len(block_failures), "dr": len(review_flags), "notes": notes, "run_id": run_id})

if block_failures:
    msg = "BLOCK checks failed: " + ", ".join(r[2] for r in block_failures) + ". No Silver tables written."
    update_run_log("FAILED", 0, 0, msg)
    raise AssertionError(msg + f" See {dq_table('dq_results')} for run_id = '{run_id}'.")

rows_out = rows_all.select("dataset_id", "measure", "psa_label", "label_as_printed", "year", "value", "value_text", "is_derived",
                           "derivation_method", "derivation_source_row_number",
                           "row_role", "unit_psgc_code", "psa_region_code", "area_psgc_code", "area_psgc_name",
                           "source_row_number", "_source_dataset_id", "_source_file", "_source_sha256", "_bronze_run_id",
                           "_silver_run_id", "_processed_at")
written = {
    TBL_MAP: write_silver(psa_map, TBL_MAP),
    TBL_ROWS: write_silver(rows_out, TBL_ROWS),
    TBL_UNIT_YEAR: write_silver(unit_year, TBL_UNIT_YEAR),
    TBL_REF: write_silver(ref_indicator, TBL_REF),
}
for tbl, n in written.items():
    print(f"✓ {silver_table(tbl)}: {n} rows")

final_status = "SUCCESS_WITH_REVIEW" if review_flags else "SUCCESS"
final_notes = (f"Silver PSA economy: {', '.join(f'{k}={v}' for k, v in written.items())}; "
               f"{len(dq_results)} checks, {len(review_flags)} for review"
               + (f" ({', '.join(r[2] for r in review_flags)})" if review_flags else "") + ".")
update_run_log(final_status, len(written), sum(written.values()), final_notes)
print(f"Run {run_id}: {final_status}\n{final_notes}")

## Preview

In [0]:
uy = spark.table(silver_table(TBL_UNIT_YEAR))
display(uy.filter(F.col("year").isin(2019, 2023)).groupBy("year")
          .agg(F.count("*").alias("units"), F.sum("gdp_total_current").alias("gdp_current_thousand_php"),
               F.sum("afs_gva_current").alias("afs_current_thousand_php")).orderBy("year"))
display(spark.table(silver_table(TBL_REF)))